# Model Serialization Formats: pickle, joblib, and ONNX

A trained Python model is just an in-memory object. To deploy it you must serialize it to a file that can be sent to a server, stored in an artifact registry, or shipped to an edge device. This notebook compares three formats: pickle, joblib, and ONNX.

**Learning objectives**
1. Explain what serialization is and why it is required for deployment.
2. Save and reload a model with both pickle and joblib, and verify predictions are preserved.
3. Understand the ONNX format and run inference with onnxruntime.
4. Choose the right format based on portability and environment requirements.

## 🔗 Where this fits

**Builds on:** Course 08 (AIAT 122) — Unit 5, lesson 03 "ONNX Model Conversion for Cross-Platform Deployment" — ONNX export and ONNX Runtime were taught there for a PyTorch model; here that same format is put head to head against pickle and joblib for an sklearn pipeline, so the choice becomes a decision rather than a default.


## 1  What is serialization?

Serialization converts a live Python object into bytes that can be written to disk or sent over a network. Without it, your model disappears when the Python process ends. The receiving system deserializes the bytes back into an object and calls `.predict()` as if nothing happened — as long as it understands the same format.


## 2  Train a classifier

We train one RandomForest on Iris data. The same trained object will be serialized three different ways.


In [1]:
# WHAT: train one reference model that every packaging format below must reproduce.
# WHY: packaging is only correct if the loaded copy predicts EXACTLY like the original —
# so we freeze reference predictions here and compare every format against them.
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import numpy as np

# float32 features now so the ONNX comparison later is apples-to-apples.
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data.astype(np.float32), iris.target,
    test_size=0.2, random_state=42
)

# Train the single source-of-truth model all formats will be compared against.
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Reference predictions — we will check that each loaded model matches these
reference_preds = clf.predict(X_test)
print(f"Test accuracy  : {clf.score(X_test, y_test):.2%}")
print(f"Reference preds (first 10): {reference_preds[:10]}")

Test accuracy  : 100.00%
Reference preds (first 10): [1 0 2 1 1 0 1 2 1 1]


## 3  pickle

`pickle` is Python's built-in serialization module. It works for any Python object, but the loaded model must be used with the same Python version and the same library versions. It is the simplest option but the least portable.


In [2]:
# WHAT: save and reload the model with pickle, then check predictions still match.
# WHY: pickle is Python's built-in serializer — simple, but Python-only and unsafe to
# load from untrusted sources, so we treat it as the baseline format.
import pickle

PICKLE_PATH = "/tmp/iris_rf.pkl"

# Save
with open(PICKLE_PATH, "wb") as f:
    pickle.dump(clf, f)

# Load into a new variable
with open(PICKLE_PATH, "rb") as f:
    clf_pkl = pickle.load(f)

# Verify
# The real test: identical predictions prove the round-trip lost nothing.
pkl_preds = clf_pkl.predict(X_test)
match = np.array_equal(pkl_preds, reference_preds)
print(f"Pickle predictions match original: {match}")
print(f"Pickle file size: {__import__('os').path.getsize(PICKLE_PATH) / 1024:.1f} KB")

Pickle predictions match original: True
Pickle file size: 170.6 KB


## 4  joblib

joblib is the format recommended by scikit-learn. It uses memory mapping for large arrays, offers optional compression, and is faster than pickle for numpy-heavy objects like forests. Same Python-version caveat applies, but it handles large model files better.


In [3]:
# WHAT: save the same model with joblib, with and without compression.
# WHY: joblib stores NumPy arrays efficiently, which is why scikit-learn recommends it
# over pickle for models full of large arrays — and compression shrinks the file further.
import joblib
import os

JOBLIB_PATH = "/tmp/iris_rf.joblib"
JOBLIB_COMPRESSED = "/tmp/iris_rf_compressed.joblib"

# Save uncompressed
joblib.dump(clf, JOBLIB_PATH)

# Save with compression level 3 (good balance of speed vs size)
joblib.dump(clf, JOBLIB_COMPRESSED, compress=3)

# Load and verify
# Round-trip check again: load the artifact and confirm predictions match.
clf_jbl = joblib.load(JOBLIB_PATH)
jbl_preds = clf_jbl.predict(X_test)
match = np.array_equal(jbl_preds, reference_preds)
print(f"Joblib predictions match original: {match}")
print(f"Joblib uncompressed : {os.path.getsize(JOBLIB_PATH) / 1024:.1f} KB")
print(f"Joblib compressed   : {os.path.getsize(JOBLIB_COMPRESSED) / 1024:.1f} KB")

Joblib predictions match original: True
Joblib uncompressed : 182.5 KB
Joblib compressed   : 25.4 KB


## 5  ONNX — Open Neural Network Exchange

ONNX is an open format that describes a model as a computation graph independent of any ML framework. Once converted, the model can run on any runtime that supports ONNX — including mobile devices, browsers, and C++ services — without Python. The `skl2onnx` library converts sklearn models; `onnxruntime` runs them.


In [4]:
# Install ONNX dependencies if not already present
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "skl2onnx", "onnxruntime"],
    capture_output=True, text=True
)
print(result.stdout[-200:] if result.stdout else "Already installed")


Already installed


In [5]:
# WHAT: convert the model to ONNX and run it with onnxruntime instead of sklearn.
# WHY: ONNX is a framework-neutral format — the serving machine no longer needs
# scikit-learn installed, which is the whole point for production and edge targets.
ONNX_PATH = "/tmp/iris_rf.onnx"

try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnxruntime as rt
    import os

    # Step 1: Convert to ONNX
    # FloatTensorType([None, 4]) means variable batch size, 4 features
    initial_type = [("float_input", FloatTensorType([None, 4]))]
    onnx_model = convert_sklearn(clf, initial_types=initial_type)

    with open(ONNX_PATH, "wb") as f:
        f.write(onnx_model.SerializeToString())

    print(f"ONNX file size: {os.path.getsize(ONNX_PATH) / 1024:.1f} KB")

    # Step 2: Run inference with onnxruntime (no sklearn needed)
    sess = rt.InferenceSession(ONNX_PATH)
    input_name = sess.get_inputs()[0].name
    label_name = sess.get_outputs()[0].name

    onnx_preds = sess.run([label_name], {input_name: X_test})[0]
    match = np.array_equal(onnx_preds, reference_preds)
    print(f"ONNX predictions match original: {match}")
    print(f"First 10 ONNX predictions: {onnx_preds[:10]}")

# Graceful fallback: if the converter is missing, show the recipe instead of crashing.
except ImportError:
    print("skl2onnx or onnxruntime not available in this environment.")
    print("ONNX conversion pattern (what the code above does):")
    print("  1. from skl2onnx import convert_sklearn")
    print("  2. onnx_model = convert_sklearn(clf, initial_types=[('float_input', FloatTensorType([None, 4]))])")
    print("  3. Save with open('model.onnx', 'wb').write(onnx_model.SerializeToString())")
    print("  4. Load with onnxruntime.InferenceSession('model.onnx')")
    print("  5. Run with sess.run([output_name], {input_name: X_test})")

ONNX file size: 78.3 KB
ONNX predictions match original: True
First 10 ONNX predictions: [1 0 2 1 1 0 1 2 1 1]


## 6  Format comparison

| Format | Python-only? | File size | Best for |
|--------|-------------|-----------|----------|
| pickle | Yes (same version) | Medium | Quick experiments, internal tools |
| joblib | Yes (same version) | Small–medium (with compression) | sklearn models in Python services |
| ONNX   | No — any ONNX runtime | Small | Cross-language, mobile, edge, high-performance inference |

**Rule of thumb:** use joblib inside a Python microservice; use ONNX when you need to cross a language boundary or deploy to a device that cannot run Python.


## Summary

Serialization converts a live model object into bytes on disk. pickle and joblib are Python-specific and easiest to use but tie you to a specific Python/sklearn version. ONNX breaks that dependency by representing the model as a framework-independent graph, enabling deployment to mobile, edge, and non-Python services. Always verify after loading: run `predict()` on a known sample and check that the output matches the original.


## Self-check

1. **What breaks if you pickle a sklearn model and load it in a newer Python version?** Think about what pickle stores alongside the model weights.
2. **Why is ONNX preferred for mobile or edge devices?** Consider that iOS and Android apps cannot run a full Python interpreter.
3. **How do you verify a model loaded correctly after deserialization?** Look at the code cells where we checked `np.array_equal(loaded_preds, reference_preds)` — what would a mismatch tell you?


## 📚 References

1. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
2. Olston, C., Fiedel, N., Gorovoy, K., et al. (2017). *TensorFlow-Serving: Flexible, High-Performance ML Serving*. NeurIPS Workshop on ML Systems. <https://arxiv.org/abs/1712.06139>
3. Paleyes, A., Urma, R.-G., & Lawrence, N. D. (2022). *Challenges in Deploying Machine Learning: A Survey of Case Studies*. ACM Computing Surveys. <https://arxiv.org/abs/2011.09926>
